# Case Study — Data Analysis and Web App

**Group number:** TBD

**Members:** TBD

# Objective

As a part of the safety inspection by the ADAC, defected seat frame "T11" was identified. The internal investigation revealed that products with ID numbers “11-213-2131-19000” to “11-213-2131-30100” were manufactured with incorrect production parameters. To ensure the customers who received the critical components are informed, we analyze the data to trace the critical parts through production, determine which high-level components and vehicles contain them and identify the customers associated with those vehicles.

The main objective of the case study is to isolate the affected products and communicate it with relevant customers. The analysis is conducted to trace the production chain from the parts to component and ultimately to vehicles as well as the corresponding registration and geographic information.


# Analysis Strategy

The analysis follows a forward traceability approach including:

- Identifying all T11  parts belonging to the critical production range.
- Determining which components contain the identified T11 parts.
- Determining which vehicles contain the affected components.
- Linking affected vehicles to registration records.
-
- Creation of one cleaned final dataset that serves as the sole input to the web application.

## 1. Data Import

Raw data was downloaded from tubCloud and extracted into `data/raw/` at the
project root (kept out of version control — see `data/raw/README.md`). We
reach it via a relative path so the notebook keeps running if someone else
drops their own copy in the same place.

In [1]:
import csv
from pathlib import Path

import pandas as pd
#TODO: Evaluation criteria said data will be placed in folder Data so "/IDA SoSe26 - Data" might need adjustment before submission

RAW_DIR = Path("../../data/IDA SoSe26 - Data")
assert RAW_DIR.exists(), f"Raw data not found at {RAW_DIR.resolve()} — see data/raw/README.md"

### 1.1 Listing available files

Before importing anything, we catalogue every file the group database
provides, grouped by category, so we can see the full scope and spot
irregularities before touching the data.

In [4]:
def list_raw_files(root: Path) -> pd.DataFrame:
    """Catalogue every file under `root`, one row per file, category = subfolder name
    (or '(root)' for files sitting directly under root, outside any category folder)."""
    rows = []
    for path in sorted(root.rglob("*")):
        if path.is_file():
            category = path.parent.name if path.parent != root else "(root)"
            rows.append({
                "category": category,
                "file": path.name,
                "extension": path.suffix,
                "size_mb": round(path.stat().st_size / 1e6, 1),
            })
    return pd.DataFrame(rows)


file_catalog = list_raw_files(RAW_DIR)
print(f"{len(file_catalog)} files across {file_catalog['category'].nunique()} categories")
file_catalog.groupby(["category", "extension"]).size()

89 files across 7 categories


category        extension
(root)                        1
                .csv          1
Einzelteil      .csv         22
                .txt         16
Fahrzeug        .csv          9
Geodaten        .csv          4
                .xlsx         1
Komponente      .csv         26
                .txt          6
Logistikverzug  .csv          2
Zulassungen     .csv          1
dtype: int64

### 1.2 Manual inspection findings

The catalogue above shows several categories mixing `.txt` and `.csv` files
(e.g. `Einzelteil/`, `Komponente/`). The extension alone does not indicate information regarding the
real format, so before writing any code we opened a byte-level sample
of at least one file per category. This surfaced three issues that a standard
`pd.read_csv`  would not have been able to parse correctly:

1. **`Einzelteil_T01.txt` has no row separator at all.** The whole file is a
   single line: fields *within* a row are separated by `" | | "`, but there
   is no character marking the end of one row and the start of the next —
   the last field of row *n* is simply followed by one bare space and then
   the (quoted) row index of row *n+1*. The header also contains three
   copies of every real column, suffixed `.x`, `.y`, and unsuffixed —
   evidence of a botched merge upstream on the source side; in the rows we
   sampled only the `.x`-suffixed columns carried real values, `.y` and the
   unsuffixed columns were all `NA`. We handle this with a dedicated parser
   (`read_glued_export` below) instead of `pd.read_csv`.
2. **`Komponente_K7.txt` looked similarly broken (unstructured might be better word for this) at first** (no `\n`
   anywhere in the file) but turned out to just use old-Mac-style
   `\r`-only line endings with tab as the field separator.
   `pd.read_csv(..., sep="\t", lineterminator="\r")` reads it directly —
   confirmed on the real file (306,490 rows x 9 columns).
3. **A duplicate file:** `Einzelteil_T23.csv` exists both directly under
   the data root and inside `Einzelteil/`. We byte-compared the two (`cmp`)
   and confirmed they are identical (172,075,946 bytes each) — a stray
   copy, not a distinct dataset. We treat the copy inside `Einzelteil/` as
   canonical and exclude the root-level one from import.

# August 12: findings

To identify further inconsistencies after the initial manual inspectoin, we inspected the relevant file from the dataset (T11) and identified a way to parse it, which is provided below in details.

1. The text files inside Einzelteil do not follow an identical structure. For instance, T01 and T11 have different formatting, namely for T11:
-  columns separated by tab, while rows are form-feed separated: \x0c, rather than a general newline character, therefore it can be read with:
*pd.read_csv(..., sep="\t", lineterminator="\x0c")*

2. Fahrzeuge_OEM1_Typ11.csv (contains information regarding where the items were produced and when) contains two redundant counter columns with names "" and "X1". To ensure removing this would indeed be harmless, we verify that it is indeed just a counter (df["Unnamed: 0"].head()
df["Unnamed: 0"].nunique()
len(df)) and  remove it by running df = df.drop(columns=["Unnamed: 0"]) in cell below.

As a result, we conclude that supplied dataset uses different conventions for separating the rows, therefore the single method cannot be reliably applied to read the files.  The loader detecs the identified format and applies parsing accordingly.



In [6]:
def read_glued_export(path: Path, delim: str, encoding: str = "latin-1") -> pd.DataFrame:
    """Parser for exports with no row separator at all: fields within a row are
    joined by `delim`, and consecutive rows are joined by a bare space with no
    delimiter of their own (see finding #1 above). Relies on two properties that
    hold for this dataset: no real field value contains a space, and every row's
    first field is a quoted row index, with row 1 always literally '"1"'."""
    text = path.read_text(encoding=encoding)
    tokens = text.split(delim)

    unglued = []
    for tok in tokens:
        if " " in tok:
            i = tok.rindex(" ")
            unglued.append(tok[:i])
            unglued.append(tok[i + 1:])
        else:
            unglued.append(tok)

    ncols = unglued.index('"1"')
    header = [h.strip('"') for h in unglued[:ncols]]
    data = unglued[ncols:]
    data = data[: len(data) - (len(data) % ncols)]  # drop a trailing partial row, if any

    rows = [data[i : i + ncols] for i in range(0, len(data), ncols)]
    df = pd.DataFrame(rows, columns=header)
    return df.apply(lambda c: c.str.strip('"') if c.dtype == "object" else c)


def load_raw_file(path: Path) -> pd.DataFrame:
    """Reads any raw file into a DataFrame, auto-detecting which of the format
    quirks from 1.2 applies. Covers the two broken patterns found so far;
    extend this if a new pattern turns up in a file we haven't sampled yet."""
    with open(path, "rb") as f:
        sample = f.read(65536)

    if b"\n" in sample:
        sample_text = sample.decode("latin-1", errors="ignore")
        try:
            sep = csv.Sniffer().sniff(sample_text, delimiters=";,\t|").delimiter
        except csv.Error:
            sep = ";"
        return pd.read_csv(path, sep=sep)

    if b"\x0c" in sample:
        return pd.read_csv(
            path,
            sep="\t",
            lineterminator="\x0c",
            encoding="latin-1",
            na_values=["NA"],
        )

    if b"\r" in sample:
        return pd.read_csv(path, sep="\t", lineterminator="\r")

    delim = " | | " if b" | | " in sample else "\t"
    return read_glued_export(path, delim)

TODO: in 1.2 parser contains a fairly strong assumption:

“no real field value contains a space”

So I would not treat it as universally safe for every dataset unless your group has verified that carefully.

### 1.3 Verifying the loader

We check `load_raw_file` against one representative file per format found
in 1.2 before relying on it more broadly.

We use *load_raw_file(path)* as adapter around several different file formats.

In [49]:
samples = {
    "Einzelteil — clean csv": RAW_DIR / "Einzelteil" / "Einzelteil_T04.csv",
    "Einzelteil — glued, no row separator": RAW_DIR / "Einzelteil" / "Einzelteil_T01.txt",
    "Komponente — \\r line endings": RAW_DIR / "Komponente" / "Komponente_K7.txt",
    "Zulassungen — clean csv": RAW_DIR / "Zulassungen" / "Zulassungen_alle_Fahrzeuge.csv",
}

for label, path in samples.items():
    df = load_raw_file(path)
    print(f"{label}: {df.shape}")
    display(df.head(3))

Einzelteil — clean csv: (1192630, 10)


,Unnamed: 0,X1,ID_T04,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung,Produktionsdatum_Origin_01011970,origin
0,1,1,4-204-2043-113,204,2043,0,NaN,0,14190,01-01-1970
1,2,2,4-202-2023-18,202,2023,0,NaN,0,14190,01-01-1970
2,3,3,4-204-2043-98,204,2043,0,NaN,0,14190,01-01-1970


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x1043c1f70>>
Traceback (most recent call last):
  File "/Users/lelat/Library/Python/3.9/lib/python/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x1043c1f70>>
Traceback (most recent call last):
  File "/Users/lelat/Library/Python/3.9/lib/python/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x1043c1f70>>
Traceback (most recent call last):
  File "/Users/lelat/Library/Python/3.9/lib/python/site-packages/ipykernel/ipkernel.py", line 775, in _c

Einzelteil — glued, no row separator: (3349745, 22)


,X1,ID_T01.x,Produktionsdatum.x,Herstellernummer.x,Werksnummer.x,Fehlerhaft.x,Fehlerhaft_Datum.x,Fehlerhaft_Fahrleistung.x,ID_T01.y,Produktionsdatum.y,...,Fehlerhaft.y,Fehlerhaft_Datum.y,Fehlerhaft_Fahrleistung.y,ID_T01,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,1,660,1-201-2011-247,2008-11-07,201,2011,0,NA,0,NA,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
1,NA,2,661,1-201-2011-429,2008-11-07,201,2011,0,NA,0,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
2,NA,NA,3,662,1-201-2011-363,2008-11-07,201,2011,1,2009-09-30,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


Komponente — \r line endings: (306490, 9)


,X1,ID_Karosserie,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung,Produktionsdatum_Origin_01011970,origin
1,1,K7-114-1142-1,114,1142,0,NaN,0,14195,01-01-1970
2,2,K7-114-1142-2,114,1142,0,NaN,0,14195,01-01-1970
3,3,K7-114-1142-5,114,1142,0,NaN,0,14196,01-01-1970


Zulassungen — clean csv: (3204104, 4)


,Unnamed: 0,IDNummer,Gemeinden,Zulassung
0,408097,11-1-11-1,DRESDEN,2009-01-01
1,408098,11-1-11-2,DRESDEN,2009-01-01
2,1,12-1-12-1,LEIPZIG,2009-01-01


# Initial EDA

Since T11 is the starting point of the traceability analysis, we start by inspecting Einzelteil_T11.txt in the Einzelteil folder. Unrelated individual-part datasets are therefore not imported at this stage.

Before transoforming anything, we conduct EDA to indeitfy types, missing values, defect distribution, etc.

In [35]:
t11_path = RAW_DIR/"Einzelteil"/"Einzelteil_T11.txt"

t11 = load_raw_file(t11_path)

print(f"Rows: {t11.shape[0]:,}")
print(f"Columns: {t11.shape[1]}")

#display the first and last 5 rows for getting the overview of entries.
print(t11.head())
print(t11.tail())

#brief summary of data
t11.info()

Rows: 2,385,260
Columns: 9
   X1           ID_T11  Herstellernummer  Werksnummer  Fehlerhaft  \
1   1  11-215-2154-102               215         2154           0   
2   2  11-212-2121-124               212         2121           0   
3   3  11-213-2131-128               213         2131           0   
4   4    11-212-2121-4               212         2121           0   
5   5   11-213-2131-70               213         2131           0   

  Fehlerhaft_Datum  Fehlerhaft_Fahrleistung  Produktionsdatum_Origin_01011970  \
1              NaN                      0.0                             14190   
2              NaN                      0.0                             14190   
3              NaN                      0.0                             14190   
4              NaN                      0.0                             14190   
5              NaN                      0.0                             14190   

       origin  
1  01-01-1970  
2  01-01-1970  
3  01-01-1970  
4  01-0

- After initial inspection, we can see that out of  2,385,260 entires, 2147106, i.e. 90% of entries in column  Fehlerhaft_Datum do not contain a value, indicating that they passed the Quality Check.

In [55]:
missing_values = (
    t11.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_values")
)

missing_values["percentage"] = (
    missing_values["missing_values"] / len(t11) * 100
).round(2)

display(missing_values)

,missing_values,percentage
Fehlerhaft_Datum,2147106,90.02
ID_T11,0,0.00
Herstellernummer,0,0.00
Werksnummer,0,0.00
Fehlerhaft,0,0.00
Fehlerhaft_Fahrleistung,0,0.00
Produktionsdatum_Origin_01011970,0,0.00
origin,0,0.00


A cross-check with the Fehlerhaft indicator shows that these missing dates correspond to parts for which no defect was recorded, while defective parts contain a defect date.

In [54]:
pd.crosstab(
    t11["Fehlerhaft"],
    t11["Fehlerhaft_Datum"].isna(),
    margins=True
)

#Fehlerhaft = 0 → mileage = 0
#Fehlerhaft = 1 → mileage > 0

Fehlerhaft_Datum,False,True,All
Fehlerhaft,,,
0,0,2147106,2147106
1,238154,0,238154
All,238154,2147106,2385260


 - To check if dataset contains duplicated entires, we also run the following cell

In [37]:
print(f"Duplicated rows: {t11.duplicated().sum():,}")

Duplicated rows: 0


After checking duplicates, we check how many distinct values we have in each rows for *Herstellernummer*, **Werksnummer* and *Fehlerhaft*

In [38]:
display(t11["Herstellernummer"].value_counts(dropna=False)) #manufacturers 213, 212 and 215 produced T11
display(t11["Werksnummer"].value_counts(dropna=False)) #werksnummer = serial number
display(t11["Fehlerhaft"].value_counts(dropna=False))

#manufacturer values, n number produced by manufacturer


Herstellernummer
213    1192630
212     715578
215     477052
Name: count, dtype: int64

Werksnummer
2131    1192630
2121     715578
2154     477052
Name: count, dtype: int64

Fehlerhaft
0    2147106
1     238154
Name: count, dtype: int64

We pay particular attention to the variables Fehlerhaft, Fehlerhaft_Datum, and Fehlerhaft_Fahrleistung. Fehlerhaft indicates whether a defect was recorded for the part, Fehlerhaft_Datum records the corresponding defect date, and Fehlerhaft_Fahrleistung represents the distance travelled when the defect was recorded.

since we know that 11-213-2131-19000 through 11-213-2131-30100 were produced using incorrect parameters, we can identify critical batch by filtering through ID range

In [39]:
print(
    t11.groupby("Fehlerhaft")[
        ["Fehlerhaft_Datum", "Fehlerhaft_Fahrleistung"]
    ].agg(["count", "min", "max"])
)

           Fehlerhaft_Datum                         Fehlerhaft_Fahrleistung  \
                      count         min         max                   count   
Fehlerhaft                                                                    
0                         0         NaN         NaN                 2147106   
1                    238154  2009-11-03  2017-11-01                  238154   

                                        
                     min           max  
Fehlerhaft                              
0               0.000000      0.000000  
1           23484.624658  34196.909589  


T11 export has an unusual extra row-index value, and X1 also appears to be a row counter. First we verify this assumption, and then we drop the column containing the duplicated information

In [40]:
print(t11.index[:10])
print(t11["X1"].head(10).tolist())

print("Rows:", len(t11))
print("Unique X1:", t11["X1"].nunique())
print("Duplicate X1:", t11["X1"].duplicated().sum())

Index([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype='int64')
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Rows: 2385260
Unique X1: 2385260
Duplicate X1: 0


X1 was verified to contain a unique sequential number for every observation (2,385,260 unique values across 2,385,260 rows), matching the source row index. It therefore represents an export-generated row counter rather than a production attribute and is removed from the analysis. (this is data cleaning step)

In [41]:
t11 = t11.reset_index(drop=True) #ensures that dataframe behaves like ordinary pandas row index
t11 = t11.drop(columns=["X1"])

We can conclude that X1 is an export-generated row counter with no production meaning.

Important column for the objectie is Fehlerhaft_Datum. To correctly analyze the values, we need to convert them to datetime type, as When pandas reads a CSV/text file, these may initially be stored as generic strings/objects rather than actual dates. This could help us further inspect earliest and latest dates of the defect by using .min() and .max() methods if necessary or extract month, year or other relevant information that is not as straightforward using strings.

"coerce" ensures that in case pandas encounters something that cannot be interpreted as a valid date, will get converted to NaT (Not a Time). The motivation for using this is presence of NA for non-defective records.

overview: : inspect raw type → convert to appropriate datetime type → verify that no valid non-missing records were lost during conversion.

In [45]:
t11["Fehlerhaft_Datum"] = pd.to_datetime(
    t11["Fehlerhaft_Datum"],
    errors="coerce"
)

#converts the column into a real pandas datetime type.

# Overview of production-date encoding:

Production date is stored as number of days since 01-01-1970, i.e.
 origin = 1970-01-01 + 14190 days = actual production date.

we will need to inspect the production date for the following reasons:
- checking when the critical batch was produced,
- detecting impossible dates,
- potentially showing the affected production period in your final analysis.

In [46]:
print(
    t11[
        ["Produktionsdatum_Origin_01011970", "origin"]
    ].head()
)

   Produktionsdatum_Origin_01011970      origin
0                             14190  01-01-1970
1                             14190  01-01-1970
2                             14190  01-01-1970
3                             14190  01-01-1970
4                             14190  01-01-1970


Check for ensuring that conversion did not introduce any accidental errors

In [47]:
original_dates = t11["Fehlerhaft_Datum"].copy()
#convert
parsed_dates = pd.to_datetime(
    original_dates,
    format="%Y-%m-%d",
    errors="coerce"
)

failed_conversion = (
    original_dates.notna()
    & parsed_dates.isna()
)

print("Non-missing dates before:", original_dates.notna().sum())
print("Dates that failed conversion:", failed_conversion.sum())

t11["Fehlerhaft_Datum"] = parsed_dates

Non-missing dates before: 238154
Dates that failed conversion: 0


In [48]:
#confirm datatype
print(t11["Fehlerhaft_Datum"].dtype)

datetime64[ns]


To identify whih Manufacturing companies and Production sites produced T11, and how many T11 parts came from each, we group the records that have the same manufacturer and serial numbers


In [58]:
manufacturer_and_serialnumbers_grouped = t11.groupby(
        ["Herstellernummer", "Werksnummer"]
    ).size().reset_index(name="count")

print(manufacturer_and_serialnumbers_grouped)

   Herstellernummer  Werksnummer    count
0               212         2121   715578
1               213         2131  1192630
2               215         2154   477052


From the criticla ID (*11-213-2131-19000*) itself, we know that:

- 11   = T11
- 213  = manufacturer
- 2131 = manufacturing plant
- 19000 = serial number

T11 was produced by multiple manufacturers. For the case study, however, manufacturer 213 and plant 2131 are of particular importance because the specified critical ID range 11-213-2131-19000 to 11-213-2131-30100 originates from this manufacturer–plant combination.


Therefore, the critical production batch specifically means:
T11 parts manufactured by manufacturer 213 at plant 2131 whose serial number is between 19000 and 30100.

# next steps:

1. clean T11
        ↓
2. extract serial number from ID_T11
        ↓
3. filter manufacturer 213
        ↓
4. filter plant 2131
        ↓
5. filter only serial 19000–30100 and store into critical_t11

And after critical_t11 exists, we can start exploring which components contain one of these critical T11 IDs.

11-213-2131-19000
        ↓
11-213-2131-30100

before doing that, we need to identify the structure of ID column to ensure looking for the values above is consistent with the format



In [65]:
t11_clean = t11.copy() #for keeping original data
t11_clean.shape

t11_clean.columns

Index(['ID_T11', 'Herstellernummer', 'Werksnummer', 'Fehlerhaft',
       'Fehlerhaft_Datum', 'Fehlerhaft_Fahrleistung',
       'Produktionsdatum_Origin_01011970', 'origin'],
      dtype='object')

In [67]:
#extract the serial number from ID_T11.
t11_clean["serial_number"] = pd.to_numeric(
    t11_clean["ID_T11"].str.rsplit("-", n=1).str[-1],
    errors="coerce"
) #creates serial number by taking serial_number = 19000 from 11-213-2131-19000

"""
Instead of comparing entire strings, we can compare the new derived field: serial_number
"""
print(t11_clean[["ID_T11", "serial_number"]].head(10))

print(
    "Failed serial number extractions:",
    t11_clean["serial_number"].isna().sum()
)

,ID_T11,serial_number
0,11-215-2154-102,102
1,11-212-2121-124,124
2,11-213-2131-128,128
3,11-212-2121-4,4
4,11-213-2131-70,70
5,11-212-2121-133,133
6,11-213-2131-229,229
7,11-212-2121-163,163
8,11-213-2131-396,396
9,11-212-2121-188,188


Failed serial number extractions: 0


Now that we validated the serial number extraction, we can create the critical batchc with serial numbers from 1900ß through 20100 by filtering manufacturer 213, plant 2131 and serial number range.

In [68]:
critical_t11 = t11_clean[
    (t11_clean["Herstellernummer"].astype(str) == "213")
    & (t11_clean["Werksnummer"].astype(str) == "2131")
    & (t11_clean["serial_number"].between(19000, 30100))
].copy()

print(f"Critical T11 records: {len(critical_t11):,}")

print(
    "Serial range:",
    critical_t11["serial_number"].min(),
    "to",
    critical_t11["serial_number"].max()
)

print(
    "Manufacturers:",
    critical_t11["Herstellernummer"].unique()
)

print(
    "Plants:",
    critical_t11["Werksnummer"].unique()
)

print(
    "Duplicate critical IDs:",
    critical_t11["ID_T11"].duplicated().sum()
)

print(critical_t11.head()) #30100 - 19000 + 1 = 11,101

expected_serials = set(range(19000, 30101))

actual_serials = set(
    critical_t11["serial_number"].dropna().astype(int)
)

missing_serials = expected_serials - actual_serials

print("Expected possible serials:", len(expected_serials))
print("Present serials:", len(actual_serials))
print("Missing serials:", len(missing_serials))

display(
    critical_t11["Fehlerhaft"]
    .value_counts(dropna=False)
    .rename("count")
)

Critical T11 records: 11,101
Serial range: 19000 to 30100
Manufacturers: [213]
Plants: [2131]
Duplicate critical IDs: 0
                  ID_T11  Herstellernummer  Werksnummer  Fehlerhaft  \
35823  11-213-2131-19101               213         2131           1   
35860  11-213-2131-19078               213         2131           0   
35953  11-213-2131-19157               213         2131           0   
35966  11-213-2131-19002               213         2131           0   
36018  11-213-2131-19229               213         2131           0   

      Fehlerhaft_Datum  Fehlerhaft_Fahrleistung  \
35823       2009-12-19                  30076.8   
35860              NaT                      0.0   
35953              NaT                      0.0   
35966              NaT                      0.0   
36018              NaT                      0.0   

       Produktionsdatum_Origin_01011970      origin  serial_number  
35823                             14236  01-01-1970          19101  
35860   

Fehlerhaft
0    9923
1    1178
Name: count, dtype: int64

In [ ]:
#The production-date conversion and some extra descriptive plots can still be added # later if useful, but they do not block the core business analysis.

#  Next, we will search for the Components that contain T11

- For tackling this business problem, we inspect the Bestandteile_Komponente_*  files and trace T11 IDs through production hierarchy. These files describe which components are installed in higher-level components. For T11, we need to check which component types contain T11 and then match with critical T11 IDs. (what is exactly matching here define more concretely)
- To do this, we inspect the schemas of the Bestandteile_Komponente_* files, that describe individual parts used in assembling a component

In [69]:
print("Missing T11 IDs:", t11_clean["ID_T11"].isna().sum())
print("Duplicate T11 IDs:", t11_clean["ID_T11"].duplicated().sum())
print("Failed serial extraction:", t11_clean["serial_number"].isna().sum())

Missing T11 IDs: 0
Duplicate T11 IDs: 0
Failed serial extraction: 0


In [70]:
component_path = RAW_DIR/"Komponente"

component_bom_files = sorted(
    component_path.glob("Bestandteile_Komponente_*")
)

print(f"files that contain relevant parts (T11): {len(component_bom_files)}")

for path in component_bom_files:
    print(path.name)

files that contain relevant parts (T11): 16
Bestandteile_Komponente_K1BE1.csv
Bestandteile_Komponente_K1BE2.csv
Bestandteile_Komponente_K1DI1.csv
Bestandteile_Komponente_K1DI2.csv
Bestandteile_Komponente_K2LE1.csv
Bestandteile_Komponente_K2LE2.csv
Bestandteile_Komponente_K2ST1.csv
Bestandteile_Komponente_K2ST2.csv
Bestandteile_Komponente_K3AG1.csv
Bestandteile_Komponente_K3AG2.csv
Bestandteile_Komponente_K3SG1.csv
Bestandteile_Komponente_K3SG2.csv
Bestandteile_Komponente_K4.csv
Bestandteile_Komponente_K5.csv
Bestandteile_Komponente_K6.csv
Bestandteile_Komponente_K7.csv


In [72]:
for path in component_bom_files[:3]:
    print("\nFILE:", path.name)

    df_header = pd.read_csv(path, nrows=0)

    print(df_header.columns.tolist())


FILE: Bestandteile_Komponente_K1BE1.csv
[';"ID_T1";"ID_T2";"ID_T3";"ID_T4";"ID_K1BE1"']

FILE: Bestandteile_Komponente_K1BE2.csv
[';"ID_T1";"ID_T2";"ID_T7";"ID_T8";"ID_K1BE2"']

FILE: Bestandteile_Komponente_K1DI1.csv
[';"ID_T1";"ID_T2";"ID_T5";"ID_T6";"ID_K1DI1"']


In [75]:
for path in component_bom_files:
    df_header = pd.read_csv(
        path,
        nrows=0,
        sep=None,
        engine="python"
    )

    print(f"\n{path.name}")
    print(df_header.columns.tolist())


Bestandteile_Komponente_K1BE1.csv
['Unnamed: 0', 'ID_T1', 'ID_T2', 'ID_T3', 'ID_T4', 'ID_K1BE1']

Bestandteile_Komponente_K1BE2.csv
['Unnamed: 0', 'ID_T1', 'ID_T2', 'ID_T7', 'ID_T8', 'ID_K1BE2']

Bestandteile_Komponente_K1DI1.csv
['Unnamed: 0', 'ID_T1', 'ID_T2', 'ID_T5', 'ID_T6', 'ID_K1DI1']

Bestandteile_Komponente_K1DI2.csv
['Unnamed: 0', 'ID_T1', 'ID_T2', 'ID_T9', 'ID_T10', 'ID_K1DI2']

Bestandteile_Komponente_K2LE1.csv
['Unnamed: 0', 'ID_T11', 'ID_T14', 'ID_T15', 'ID_K2LE1']

Bestandteile_Komponente_K2LE2.csv
['Unnamed: 0', 'ID_T16', 'ID_T19', 'ID_T20', 'ID_K2LE2']

Bestandteile_Komponente_K2ST1.csv
['X1', 'X', 'ID_T11', 'ID_T12', 'ID_T13', 'ID_K2ST1']

Bestandteile_Komponente_K2ST2.csv
['Unnamed: 0', 'ID_T16', 'ID_T17', 'ID_T18', 'ID_K2ST2']

Bestandteile_Komponente_K3AG1.csv
['X1', 'ID_T21', 'ID_T24', 'ID_T25', 'ID_K3AG1']

Bestandteile_Komponente_K3AG2.csv
['X1', 'ID_T21', 'ID_T24', 'ID_T27', 'ID_K3AG2']

Bestandteile_Komponente_K3SG1.csv
['X1', 'ID_T21', 'ID_T22', 'ID_T23', 'I

In [74]:
for path in component_bom_files:
    df_header = pd.read_csv(
        path,
        nrows=0,
        sep=None,
        engine="python" #for automatic delimiter detection
    )

    if "ID_T11" in df_header.columns:
        print(f"T11 found in: {path.name}")
        print(df_header.columns.tolist())

T11 found in: Bestandteile_Komponente_K2LE1.csv
['Unnamed: 0', 'ID_T11', 'ID_T14', 'ID_T15', 'ID_K2LE1']
T11 found in: Bestandteile_Komponente_K2ST1.csv
['X1', 'X', 'ID_T11', 'ID_T12', 'ID_T13', 'ID_K2ST1']


The results of running the cell above indicates that Bestandteile_Komponente_K2LE1.csv contains specific ID_T11 parts and Bestandteile_Komponente_K2ST1 as well.

T11
├── can be part of K2LE1
└── can be part of K2ST1

In [76]:
k2le1_bom = load_raw_file(
    RAW_DIR / "Komponente" / "Bestandteile_Komponente_K2LE1.csv"
)

k2st1_bom = load_raw_file(
    RAW_DIR / "Komponente" / "Bestandteile_Komponente_K2ST1.csv"
)

print("K2LE1:", k2le1_bom.shape)
display(k2le1_bom.head())

print("K2ST1:", k2st1_bom.shape)
display(k2st1_bom.head())

K2LE1: (477052, 5)


,Unnamed: 0,ID_T11,ID_T14,ID_T15,ID_K2LE1
0,1,11-215-2154-70,14-214-2142-8,15-209-2093-12,K2LE1-109-1091-1
1,2,11-215-2154-102,14-214-2142-11,15-209-2093-20,K2LE1-109-1091-2
2,3,11-212-2121-7,14-213-2132-1,15-210-2101-2,K2LE1-109-1091-3
3,4,11-212-2121-61,14-213-2132-5,15-210-2101-21,K2LE1-109-1091-4
4,5,11-212-2121-95,14-213-2132-6,15-210-2101-24,K2LE1-109-1091-5


K2ST1: (1908208, 6)


,X1,X,ID_T11,ID_T12,ID_T13,ID_K2ST1
0,1,1,11-213-2131-128,12-212-2121-15,13-209-2091-25,K2ST1-109-1092-1
1,2,2,11-213-2131-134,12-212-2121-28,13-209-2092-7,K2ST1-109-1092-2
2,3,3,11-212-2121-107,12-212-2121-73,13-209-2092-69,K2ST1-109-1092-3
3,4,4,11-213-2131-182,12-212-2122-41,13-209-2092-77,K2ST1-109-1092-4
4,5,5,11-212-2121-124,12-212-2122-71,13-209-2092-86,K2ST1-109-1092-5


we select teh relevant columns, since from preview above we see that ID_T11 and ID_K2LE1 amd ST! are relevant ones

This indicates that T11 parts can be installed in components of type K2LE1 or K2ST1. The corresponding ID_K2LE1 and ID_K2ST1 columns identify the higher-level components containing each individual T11 part..

To further trace the hierarchy, only those 2 component files need to be analyzed.

In [77]:
k2le1_trace = k2le1_bom[
    ["ID_T11", "ID_K2LE1"]
].copy()

k2st1_trace = k2st1_bom[
    ["ID_T11", "ID_K2ST1"]
].copy()

critical_ids = set(critical_t11["ID_T11"])

affected_k2le1 = k2le1_trace[
    k2le1_trace["ID_T11"].isin(critical_ids)
].copy()

affected_k2st1 = k2st1_trace[
    k2st1_trace["ID_T11"].isin(critical_ids)
].copy()

print("Affected K2LE1 rows:", len(affected_k2le1))
print("Affected K2LE1 components:", affected_k2le1["ID_K2LE1"].nunique())

print("Affected K2ST1 rows:", len(affected_k2st1))
print("Affected K2ST1 components:", affected_k2st1["ID_K2ST1"].nunique())

display(affected_k2le1.head())
display(affected_k2st1.head())

Affected K2LE1 rows: 2201
Affected K2LE1 components: 2201
Affected K2ST1 rows: 8900
Affected K2ST1 components: 8900


,ID_T11,ID_K2LE1
7545,11-213-2131-19061,K2LE1-109-1091-7546
7546,11-213-2131-19134,K2LE1-109-1091-7547
7547,11-213-2131-19141,K2LE1-109-1091-7548
7551,11-213-2131-19447,K2LE1-109-1091-7552
7552,11-213-2131-19466,K2LE1-109-1091-7553


,ID_T11,ID_K2ST1
28661,11-213-2131-19005,K2ST1-109-1092-28662
28748,11-213-2131-19171,K2ST1-109-1092-28749
28756,11-213-2131-19059,K2ST1-109-1092-28757
28838,11-213-2131-19039,K2ST1-109-1092-28839
28931,11-213-2131-19043,K2ST1-109-1092-28932


Because the component IDs have different column names, standardize them first:


In [79]:
affected_k2le1 = affected_k2le1.rename(
    columns={"ID_K2LE1": "component_id"}
)
affected_k2le1["component_type"] = "K2LE1"

affected_k2st1 = affected_k2st1.rename(
    columns={"ID_K2ST1": "component_id"}
)
affected_k2st1["component_type"] = "K2ST1"

#and then combine them

affected_components = pd.concat(
    [
        affected_k2le1[["ID_T11", "component_id", "component_type"]],
        affected_k2st1[["ID_T11", "component_id", "component_type"]],
    ],
    ignore_index=True
)

print("Affected component relationships:", len(affected_components))
print("Unique critical T11 IDs matched:", affected_components["ID_T11"].nunique())
print("Unique affected components:", affected_components["component_id"].nunique())

display(affected_components.head())

Affected component relationships: 11101
Unique critical T11 IDs matched: 11101
Unique affected components: 11101


,ID_T11,component_id,component_type
0,11-213-2131-19061,K2LE1-109-1091-7546,K2LE1
1,11-213-2131-19134,K2LE1-109-1091-7547,K2LE1
2,11-213-2131-19141,K2LE1-109-1091-7548,K2LE1
3,11-213-2131-19447,K2LE1-109-1091-7552,K2LE1
4,11-213-2131-19466,K2LE1-109-1091-7553,K2LE1


Check whether every critical T11 was actually found in one of these component parts lists:

In [81]:
matched_t11_ids = set(affected_components["ID_T11"])

unmatched_t11_ids = critical_ids - matched_t11_ids

print("Critical T11 IDs:", len(critical_ids))
print("Matched to a component:", len(matched_t11_ids))
print("Not matched to a component:", len(unmatched_t11_ids))

"""
paraphrase lateR:

All 11,101 T11 parts belonging to the critical production batch were successfully matched to a higher-level component. Of these, 2,201 were installed in K2LE1 components and 8,900 in K2ST1 components. No critical T11 IDs remained unmatched. Furthermore, the number of unique T11 IDs, component relationships, and unique component IDs is identical (11,101), indicating a one-to-one T11-to-component relationship in the observed parts-list data.

"""

Critical T11 IDs: 11101
Matched to a component: 11101
Not matched to a component: 0


'\nparaphrase lateR:\n\nAll 11,101 T11 parts belonging to the critical production batch were successfully matched to a higher-level component. Of these, 2,201 were installed in K2LE1 components and 8,900 in K2ST1 components. No critical T11 IDs remained unmatched. Furthermore, the number of unique T11 IDs, component relationships, and unique component IDs is identical (11,101), indicating a one-to-one T11-to-component relationship in the observed parts-list data.\n\n'